In [12]:
from typing import Any

from pydantic import BaseModel
from unstructured.partition.pdf import partition_pdf



In [13]:
images_path = './images'
raw_pdf_elements = partition_pdf(
    filename="statement_of_changes.pdf",
    # filename="Wi-Fi 6(802.11ax)解析23：QTP（Quiet Time Period）和QP.pdf",
    strategy="hi_res",
    extract_images_in_pdf=True,
    extract_image_block_types=["Image", "Figure"],
    infer_table_structure=True,
    include_metadata=True,
    include_page_breaks=True,
    # 改用支援的策略，例如 "by_title"
    # chunking_strategy="by_title",
    max_characters=1500,         # 降低每個段落的最大字元數
    new_after_n_chars=1400,      # 降低觸發新段落的字元數
    combine_text_under_n_chars=500,  # 降低合併門檻，使分段更細
    image_output_dir_path=images_path,
)

# for i in raw_pdf_elements:
#     print(i.category)

In [14]:
category_counts = {}

for element in raw_pdf_elements:
    category = str(type(element))
    if category in category_counts:
        category_counts[category] += 1
    else:
        category_counts[category] = 1

unique_categories = set(category_counts.keys())
category_counts

{"<class 'unstructured.documents.elements.Title'>": 12,
 "<class 'unstructured.documents.elements.NarrativeText'>": 7,
 "<class 'unstructured.documents.elements.Header'>": 1,
 "<class 'unstructured.documents.elements.Table'>": 4,
 "<class 'unstructured.documents.elements.FigureCaption'>": 1,
 "<class 'unstructured.documents.elements.ListItem'>": 5,
 "<class 'unstructured.documents.elements.PageBreak'>": 2,
 "<class 'unstructured.documents.elements.Text'>": 2}

In [8]:
class Element(BaseModel):
    type: str
    text: Any

table_elements = []
text_elements = []
for element in raw_pdf_elements:
    if "unstructured.documents.elements.Table" in str(type(element)):
        table_elements.append(Element(type="table", text=str(element)))
    elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
        text_elements.append(Element(type="text", text=str(element)))

In [40]:
print(len(table_elements))
print(len(text_elements))

1
4


In [42]:
table_elements[0]

Element(type='table', text='1.Title of Security 2. Trans. Date |2A. Deemed 3. Trans. Code 4. Securities Acquired (A) |5. Amount of Securities Beneficially Owned 6. 7. Nature (Instr. 3) Execution __ | (Instr. 8) or Disposed of (D) Following Reported Transaction(s) Ownership] of Indirect Date, if any (Instr. 3, 4 and 5) (Instr. 3 and 4) Form: Beneficial Direct (D) | Ownership or Indirect | (Instr. 4) (A) or (D (Instr. Code | V Amount} (D) _ | Price 4) The Welch- Common Stock 10/6/2023 G 2300] dD {sof 7,106 I Drell 2009 Revocable Trust 2) ‘Common Stock 26,518 D x Rose I Common Stock 68 I Welch x Joseph Common Stock 68 I Welch ~ Cornelia Common Stock 68 I Welch')

In [43]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

In [52]:
prompt_text = """
  You are responsible for concisely summarizing table or text chunk:

  {element}
"""
prompt = ChatPromptTemplate.from_template(prompt_text)
summarize_chain = {"element": lambda x: x} | prompt | ChatOpenAI(temperature=0, model="gpt-4") | StrOutputParser()

In [53]:
tables = [i.text for i in table_elements]
table_summaries = summarize_chain.batch(tables, {"max_concurrency": 5})

texts = [i.text for i in text_elements]
text_summaries = summarize_chain.batch(texts, {"max_concurrency": 5})

In [54]:
table_summaries

['The table provides information about security transactions. It includes the title of the security, transaction date, transaction code, the number of securities acquired or disposed of, the amount of securities beneficially owned following reported transactions, and the nature of ownership. The specific transactions listed are for common stock. The Welch-Drell 2009 Revocable Trust disposed of 2300 units on 10/6/2023. Other transactions include 26,518 units disposed by Rose, and 68 units each by Joseph and Cornelia Welch.']

In [1]:
text_summaries

NameError: name 'text_summaries' is not defined

In [ ]:
import uuid

from langchain.embeddings import OpenAIEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.schema.document import Document
from langchain.storage import InMemoryStore
from langchain.vectorstores import Chroma

id_key = "doc_id"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings()),
    docstore=InMemoryStore(),
    id_key=id_key,
)

# Add texts
doc_ids = [str(uuid.uuid4()) for _ in texts]
summary_texts = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(text_summaries)
]
retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, texts)))

# Add tables
table_ids = [str(uuid.uuid4()) for _ in tables]
summary_tables = [
    Document(page_content=s, metadata={id_key: table_ids[i]})
    for i, s in enumerate(table_summaries)
]
retriever.vectorstore.add_documents(summary_tables)
retriever.docstore.mset(list(zip(table_ids, tables)))